# Asset selling: value iteration

We wish to sell a used car by choosing when to accept an offer. The car is
exactly two years old. Offers for this model at this age follow a discrete
distribution: in the initial month the offer $\bar w_0$ takes values in
$\{c_1,\dots,c_C\}$ with $\text{Prob}[\bar w_0=c_i]=p_i$, and
$\sum_i p_i = 1-p_0$ where $p_0$ is the probability of no offer at all. The
car depreciates at a rate $\alpha$ per month, so the offer $k$ months later
is $\bar w_k$, and the *scaled* offer $w_k = \bar w_k/\alpha^k$ has the same
distribution as $\bar w_0$.

The decision $u_k\in\{0,1\}$ is to wait or to sell. The state $x_k$ is the
most recent scaled offer, or the symbol $N$ if there was no offer, or $T$ if
the car has already been sold. After about ten years the residual value is
negligible, so an infinite horizon with discounting is a fair idealisation:

$$\max_\pi \ \sum_{k=0}^{\infty}\alpha^kg(x_k,u_k), \qquad g(x_k,1)=x_k, \quad g(x_k,0)=0, \quad g(N,\cdot)=g(T,\cdot)=0 .$$

Value iteration reduces to a single scalar comparison per state -- take what
is on the table now, or hold out for the discounted expected value of
waiting:

$$J_{k+1}(x_k)=\max\Big\{x_k,\ \alpha\sum_{i=1}^{C}p_iJ_k(c_i)\Big\}, \qquad J_k(N)=J_k(T)=0 .$$

Note the direction: this is the one example in Unit 10 stated as a
*maximisation* of revenue rather than a minimisation of cost. The two are
the same problem up to a sign.

This notebook reproduces the two figures in the unit, and then runs the
iteration to convergence, which the figures do not show.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

In [ ]:
# problem data: 101 possible offers from 10000 to 20000 euros, with a
# roughly normal distribution centred on 15000
C     = 101
alpha = 0.99
c     = 10000 + np.arange(C) * 100
var   = 2000.0 ** 2
p     = np.exp(-1 / (2 * var) * (c - c[C // 2]) ** 2)
p     = p / p.sum()

print(f"offers from {c[0]} to {c[-1]} euros, centred on {c[C // 2]}")

In [ ]:
def value_iteration(c, p, alpha, tol=1e-3):
    """Value iteration for the asset selling problem.

    Returns the converged value function, and the full history of the
    iterates (values and policies), one column per iteration.
    """
    J = np.zeros(len(c))
    J_hist, u_hist = [], []
    while True:
        wait  = alpha * (p @ J)              # value of declining this offer
        J_new = np.maximum(wait, c)          # ... versus taking it
        u     = (c >= wait).astype(int)      # 1 = sell, 0 = wait
        if np.max(np.abs(J_new - J)) < tol:
            return J, np.array(J_hist).T, np.array(u_hist).T
        J = J_new
        J_hist.append(J.copy())
        u_hist.append(u.copy())


J, J_hist, u_hist = value_iteration(c, p, alpha)
print(f"{J_hist.shape[1]} iterations to converge")

In [ ]:
# the first ten iterates -- the two figures in the unit
N = 10
fig, ax = plt.subplots(1, 2)
for k in range(N):
    shade = str(1 - (k + 1) / N)             # k=0 light, k=9 dark
    ax[0].plot(c, J_hist[:, k], color=shade, label=f"k={k}")
    ax[1].plot(c, u_hist[:, k], color=shade, label=f"k={k}")
ax[0].set_xlabel("i"); ax[0].set_ylabel("$J_k(i)$"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("i"); ax[1].set_ylabel(r"$\mu_k(i)$"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
def threshold(u, c):
    """The offer at which the policy switches from wait to sell."""
    return c[np.argmax(u == 1)]


for k in range(N):
    print(f"  k={k}:  threshold {threshold(u_hist[:, k], c)} euros")

print(f"\nconverged:  wait value {alpha * (p @ J):.1f},"
      f"  threshold {threshold((c >= alpha * (p @ J)).astype(int), c)} euros")

The policy is a **threshold rule**: sell if the scaled offer exceeds a
fixed number, otherwise wait. It is stationary -- it does not depend on
time -- which is what the infinite horizon buys us. A finite-horizon
version of the same problem has a threshold that falls as the deadline
approaches.

Two numbers are worth separating. The tenth iterate, which is the last
curve plotted above and the one quoted in the lecture, has a threshold of
about $16\,700$ euros. The *converged* policy has a threshold of
$16\,900$, with a wait value of $16\,857.9$. Both are well above the mean
offer of $15\,000$: waiting has option value, and the threshold is the
price at which that value is exactly used up.

`A_10_stoppingtimeQLearning.ipynb` solves this same problem *without* the
model -- without $p$ -- and recovers the same number.